# EDA - Procena populacije / izgrađenosti iz satelitskih snimaka (Srbija)

Provera da li podaci **pasuju** za DL projekat (CNN + transfer learning).
Izvori (svi besplatni): **GeoSrbija RPJ** (geometrija naselja/opština/krugova), **RZS Popis 2022**
(populacija po naseljima), **Overture** (footprinti), **Sentinel-2 L2A** preko openEO (snimci).

**Pillari koje proveravamo:** geometrija, labele (populacija), footprinti (izgrađenost), snimci.

In [ ]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import matplotlib.pyplot as plt
%matplotlib inline

# koren repoa (sadrzi core/) na sys.path, pa sve putanje iz scripts.config
import sys
_koren = os.getcwd()
while not os.path.isdir(os.path.join(_koren, "core")) and os.path.dirname(_koren) != _koren:
    _koren = os.path.dirname(_koren)
if not os.path.isdir(os.path.join(_koren, "core")):
    raise RuntimeError("nema core/ - kloniraj ceo repo, ne samo notebook")
if _koren not in sys.path:
    sys.path.insert(0, _koren)
from scripts import config
from scripts.preprocessing import build_labels

NAS, OPS, PK = config.NASELJA_GPKG, config.OPSTINE_GPKG, config.POPISNI_KRUG
config.obezbedi(config.RESULTS, config.FIGURES)   # terminalne tabele i slike

## 1. Prostorne jedinice (RPJ)
Broj jedinica, CRS, površina (iz geometrije), MAUP rizik. Napomena: RPJ OpenData export ima SAMO geometriju, sve statističke kolone su nule.

In [ ]:
nas = gpd.read_file(NAS)
ops = pyogrio.read_dataframe(OPS, read_geometry=False)
pk  = pyogrio.read_dataframe(PK,  read_geometry=False)
nas["area_km2"] = nas.geometry.area / 1e6
print("naselje:", len(nas), "| opštine sa naseljima:", nas.opstina_maticni_broj.nunique(), "| CRS:", nas.crs)
print("opstine rows:", len(ops), "| popisni krug:", len(pk))
print("area km2 -> total %.0f (=Srbija bez KiM)  p50 %.1f  max %.1f" % (
    nas.area_km2.sum(), nas.area_km2.median(), nas.area_km2.max()))
print("\nMAUP rizik (najveća naselja = jedna labela preko ogromne površine):")
display(nas.nlargest(5, "area_km2")[["naselje_ime", "opstina_ime", "area_km2"]])

## 2. Labele: populacija (RZS 2022) -> geometrija
RZS xlsx je hijerarhijski po imenu (bez šifre), pa se spaja u 3 koraka: par (opština, naselje) → jedinstveno ime naselja → ručni crosswalk za pravopisne varijante. Logika je u `scripts/preprocessing/build_labels.py` i odatle se uvozi; ovde se samo proverava rezultat, `naselje_pop_final.csv` piše skripta. **Validacija: zbir = nacionalni total 6.646.833.**

In [ ]:
labele = build_labels.spoji_labele()          # ista logika koju pokrece i pipeline

spojeno, zbir = int(labele["pop"].notna().sum()), int(labele["pop"].sum())
print("spojeno %d/%d = %.2f%%" % (spojeno, len(labele), spojeno / len(labele) * 100))
print("zbir %d  (RZS nacionalni total 6646833)  -> %s" % (
    zbir, "zbir se slaze" if zbir == 6646833 else "NE SLAZE SE"))

KORAK = {0: "nespojeno", 1: "par (opstina, naselje)", 2: "jedinstveno ime", 3: "rucni crosswalk"}
display(labele.stage.value_counts().sort_index().rename(index=KORAK).to_frame("naselja"))
display(labele.loc[labele["pop"].isna(), ["opstina_ime", "naselje_ime"]])
print("nespojeno = RPJ artefakt bez popisnog para (build_labels.BEZ_POPISA)")

## 3. Distribucija populacije
Jaka desna iskošenost → `log1p(pop)` cilj. Depopulacija: naselja sa ~0 stanovnika.

In [ ]:
p = labele["pop"].dropna()
print("median %d  mean %.0f  max %d  zeros %d  <=10 %d" % (
    p.median(), p.mean(), p.max(), (p == 0).sum(), (p <= 10).sum()))
print("skew raw %.2f  ->  log1p %.2f" % (p.skew(), np.log1p(p).skew()))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(p, bins=60, color="#c44")
ax[0].set_title("population raw (skew %.1f)" % p.skew())
ax[1].hist(np.log1p(p), bins=60, color="#48c")
ax[1].set_title("log1p(pop) ~ normalno")
plt.tight_layout()
plt.show()

## 4. Footprinti (Overture): pokrivenost sela
Provera da li ML footprinti pokrivaju sela (rizik depopulacije). Učitava keširani `results/rural_footprints.csv` (generiše ga `scripts/footprint/coverage.py`).

In [ ]:
if os.path.exists(config.RURAL_FOOTPRINTS):
    rr = pd.read_csv(config.RURAL_FOOTPRINTS)
    display(rr)
    print("zero-coverage:", int((rr.buildings==0).sum()), "/", len(rr),
          "| izvori:", rr.top_source.value_counts().to_dict())
    print("NALAZ: footprinti opstaju i posle depopulacije -> izgradjenost != populacija")
    print("  (npr. ДЕЈАНОВАЦ: pop 3, ~100 zgrada). Model mora citati ZAUZETOST iz slike, ne samo broj krovova.")
else:
    print("Pokreni python -m scripts.footprint.coverage da generises", config.RURAL_FOOTPRINTS)

## 5. Sentinel-2 sample (openEO)
Dokaz pipeline-a: bezoblačni letnji medijan, 4 opsega, isečak oko centroida. Učitava keširani GTiff; ako ga nema, povlači (auth je tih ako je token keširan).

In [ ]:
import rasterio

UZORAK_NASELJE = "НОВИ САД"
POLA_STRANE_M = 1115          # ~224 px na 10 m
OPSEZI_RGB = ["B02", "B03", "B04", "B08"]

TIF = os.path.join(config.SENTINEL_DIR, "novi_sad_s2.tiff")
if not os.path.exists(TIF):
    import openeo
    veza = openeo.connect("openeo.dataspace.copernicus.eu")
    veza.authenticate_oidc()

    geometrija = gpd.read_file(NAS)
    centar = geometrija[geometrija.naselje_ime == UZORAK_NASELJE].iloc[0].geometry.centroid
    opseg = {"west": centar.x - POLA_STRANE_M, "south": centar.y - POLA_STRANE_M,
             "east": centar.x + POLA_STRANE_M, "north": centar.y + POLA_STRANE_M,
             "crs": "EPSG:32634"}
    kocka = veza.load_collection(
        "SENTINEL2_L2A", spatial_extent=opseg,
        temporal_extent=["2024-05-01", "2024-09-30"], bands=OPSEZI_RGB,
        max_cloud_cover=30).reduce_dimension(dimension="t", reducer="median")
    kocka.download(TIF, format="GTiff")

with rasterio.open(TIF) as dataset:
    arr = dataset.read().astype("float32")
print("shape", arr.shape, "| nan%", round(float(np.isnan(arr).mean() * 100), 1),
      "| crs EPSG:32634")


def rastegni(kanal):
    """Kontrastno rastezanje na 2-98 percentil, za prikaz."""
    lo, hi = np.nanpercentile(kanal, 2), np.nanpercentile(kanal, 98)
    return np.clip((kanal - lo) / (hi - lo + 1e-6), 0, 1)


rgb = np.dstack([rastegni(arr[2]), rastegni(arr[1]), rastegni(arr[0])])
plt.figure(figsize=(4.5, 4.5))
plt.imshow(rgb)
plt.axis("off")
plt.title("Novi Sad - S2 RGB median 2024 (224px @10m)")
plt.show()

## 6. Verdikt

| Pillar | Status | Dokaz |
|---|---|---|
| Geometrija | OK | 4.721 naselja / 168 opština, EPSG:32634, površina = Srbija bez KiM |
| Labele | OK | RZS join 99.98%, zbir = 6.646.833 (nacionalni total) |
| Footprinti | OK | Overture pokriva i sela (ML), 0 praznih; izgrađenost != populacija u depopulaciji |
| Snimci | OK | openEO S2 isečak, bezoblačan medijan, bez rupa |

**Podaci na disku ~0.7 GB.** Pun MVP (isečci 1 god, 6 opsega) ~5-8 GB.

**Sledeće (Faza 2, gradnja dataseta):** centroid+label tabela -> footprinti po naselju (izgrađenost cilj) ->
Sentinel isečci loop (4.720 x 224px x 6 opsega) -> GroupKFold po opštini.